# 03 -- Signal pipeline

The decoupled steps and their permutations.

In [1]:
%matplotlib inline

import numpy as np
from asdea_sensors import SensorDataset
from asdea_sensors.config import settings

# The example data folder (edit to your path if needed).
DATA = r"C:\Users\ppala\Desktop\02_31MAY2026"

# A short window keeps every example fast (one ~10 min file).
WSTART, WLEN = "2026-05-31 15:00:00", "10min"

In [2]:
ds = SensorDataset(DATA, verbose=True)
ds.devices

------------------------------------------------------------
SensorDataset
------------------------------------------------------------
path        : C:\Users\ppala\Desktop\02_31MAY2026
files       : 32
time span   : 2026-05-31 14:52:12  ->  2026-05-31 20:02:13
duration    : 18601.0 s
devices     : MNAT0031, MNAT0034, MOF00134, MOF00135, MOF00136
fs / dt     : 252.5885 Hz / 0.003959 s
------------------------------------------------------------
axes (per sensor):
  MNAT0031   -> (3, 1, 5)
  MNAT0034   -> (3, 1, 5)
  MOF00134   -> (0, 1, 2)
  MOF00135   -> (3, 1, 5)
  MOF00136   -> (3, 1, 5)
------------------------------------------------------------
on-disk size: 782.17 MB
RAM         : used 33.25 GB / avail 0.27 GB (99%)
------------------------------------------------------------


['MNAT0031', 'MNAT0034', 'MOF00134', 'MOF00135', 'MOF00136']

## Raw signal

In [3]:
sig = ds.MOF00135.window(WSTART, WLEN).signal(components="all")
print("n =", sig.n, " axes =", sig.axes)

[signal] MOF00135 n=149240 dt=0.004021 comps=all
n = 149240  axes = (0, 1, 2)


C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\src\asdea_sensors\core\h5_reader.py:45: UserWarning: H5Reader: axes (3, 1, 5) for 'MOF00135' exceed the 3 available columns; falling back to (0, 1, 2)
  warnings.warn(


## Each step

In [4]:
b = ds.MOF00135.window(WSTART, WLEN).baseline()
f = b.filter(0.1, 24.9)
d = f.derive()
out = d.signal(components="x")
print("vel filled:", out.vel_x is not None, " disp filled:", out.disp_x is not None)

[signal] MOF00135 n=149240 dt=0.004021 comps=x
vel filled: True  disp filled: True


C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\src\asdea_sensors\derive\filters.py:49: UserWarning: obspy not available; falling back to the scipy engine
  warnings.warn("obspy not available; falling back to the scipy engine")


## Permutations of order

In [5]:
a = ds.MOF00135.window(WSTART, WLEN).baseline().filter(0.1, 24.9).derive().signal()
b = ds.MOF00135.window(WSTART, WLEN).filter(0.1, 24.9).baseline().derive().signal()
c2 = ds.MOF00135.window(WSTART, WLEN).filter(0.5, 12.0, engine="scipy").derive().signal()
print("three pipelines built:", a.n, b.n, c2.n)

[signal] MOF00135 n=149240 dt=0.004021 comps=all


[signal] MOF00135 n=149240 dt=0.004021 comps=all
[signal] MOF00135 n=149240 dt=0.004021 comps=all
three pipelines built: 149240 149240 149240


## Resample then process

In [6]:
s100 = ds.MOF00135.window(WSTART, WLEN).resample(fs=100).filter(0.1, 24.9).signal(components="x")
print("resampled dt =", round(s100.dt, 5), " fs =", round(s100.fs, 1))

[signal] MOF00135 n=60006 dt=0.010000 comps=x
resampled dt = 0.01  fs = 100.0
